In [1]:
import os
import glob
import pandas as pd

base_dir = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports"

# Adjust the pattern if filenames have a specific pattern, e.g. "*_daily.csv"
csv_files = glob.glob(os.path.join(base_dir, "*.csv"))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df["source_file"] = os.path.basename(f)  # optional, to track origin
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)
all_data

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner,FileDate,source_file,ensemble_45d,Confidence: 45-Day,Frequency (1d),Probability: 1-Day,Confidence: 1-Day
0,102.129.153.158,0,0.0,1.0,6.88%,7-Day: Low confidence,13.68%,14-Day: Low confidence,68.66%,30-Day: Possibly active,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
1,102.129.153.43,0,0.0,0.0,0.38%,7-Day: Low confidence,0.93%,14-Day: Low confidence,18.11%,30-Day: Low confidence,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
2,102.129.153.71,0,0.0,2.0,12.39%,7-Day: Low confidence,24.8%,14-Day: Low confidence,86.71%,30-Day: Highly likely,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.01%,14-Day: Low confidence,0.17%,30-Day: Low confidence,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
4,103.133.107.28,0,0.0,1.0,9.14%,7-Day: Low confidence,59.14%,14-Day: Possibly active,73.35%,30-Day: Possibly active,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363758,teletype.in,0,0.0,3.0,13.68%,7-Day: Low confidence,67.53%,14-Day: Possibly active,92.48%,30-Day: Highly likely,VA,20260312,full_daily_report_20260312.csv,77.35%,45-Day: Highly likely,0.0,nan%,1-Day: Low confidence
2363759,tester.com,0,0.0,0.0,0.03%,7-Day: Low confidence,0.02%,14-Day: Low confidence,0.42%,30-Day: Low confidence,VA,20260312,full_daily_report_20260312.csv,13.31%,45-Day: Low confidence,0.0,nan%,1-Day: Low confidence
2363760,www.citiquartz.com,0,0.0,1.0,5.02%,7-Day: Low confidence,11.41%,14-Day: Low confidence,66.85%,30-Day: Possibly active,VA,20260312,full_daily_report_20260312.csv,52.31%,45-Day: Possibly active,0.0,nan%,1-Day: Low confidence
2363761,www.deepseek.com,0,1.0,17.0,91.65%,7-Day: Possibly active,99.69%,14-Day: Highly likely,99.98%,30-Day: Highly likely,VA,20260312,full_daily_report_20260312.csv,96.33%,45-Day: Highly likely,0.0,nan%,1-Day: Low confidence


In [2]:
# Clean, convert, and aggregate in one step

# Rename 45-day probability column if present
if "ensemble_45d" in all_data.columns:
    all_data = all_data.rename(columns={"ensemble_45d": "Probability: 45-Day"})

# Drop confidence columns and other unwanted fields
cols_to_drop = [c for c in all_data.columns if "confidence" in c.lower()]
cols_to_drop += ["source_file", "FileDate", "Partner", "Observed Today"]
# Remove duplicates while preserving order
cols_to_drop = list(dict.fromkeys(cols_to_drop))
all_data = all_data.drop(columns=cols_to_drop, errors="ignore")

# Convert probability columns from strings like "12.3%" to numeric 0–1 floats
prob_cols = [c for c in all_data.columns if "Probability" in c]
for c in prob_cols:
    all_data[c] = pd.to_numeric(
        all_data[c].astype(str).str.rstrip("%"), errors="coerce"
    ) / 100.0

# Group by Indicator and take average of numeric columns
grouped = all_data.groupby("Indicator", as_index=False).mean(numeric_only=True)

# Round frequency columns to nearest whole number
freq_cols = [c for c in grouped.columns if c.startswith("Frequency ")]
for c in freq_cols:
    grouped[c] = grouped[c].round(0).astype("Int64")

# Convert probabilities (0–1) to percentages with 2 decimals
prob_cols_grouped = [c for c in grouped.columns if "Probability" in c]
for c in prob_cols_grouped:
    grouped[c] = (grouped[c] * 100).round(2)    

grouped

,Indicator,Frequency (7d),Frequency (30d),Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day,Frequency (1d),Probability: 1-Day
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0,1,11.20,22.60,32.99,36.37,0,0.42
1,1-you.njalla.no,0,2,24.55,41.27,63.04,49.73,0,9.32
2,1.192.18.4,0,1,15.94,29.39,46.53,43.33,0,0.01
3,1.204.166.3,0,1,12.10,24.36,48.52,41.00,0,0.08
4,1.234.83.26,3,13,66.81,75.04,88.05,75.68,0,34.68
...,...,...,...,...,...,...,...,...,...
5804,ymath.yonsei.ac.kr,0,0,0.17,1.02,2.64,12.94,0,0.02
5805,yotpo-static.com,0,0,1.39,1.69,5.26,13.81,0,0.04
5806,yourpensionmeeting.com,3,12,57.07,66.88,78.93,66.80,0,32.59
5807,zerocap.com,0,0,0.00,0.00,0.00,NaN,<NA>,NaN


In [3]:
tas_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"

# Read the first sheet (sheet index 0) into a DataFrame
tas_df = pd.read_excel(tas_path, sheet_name=0)
tas_df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,1.234.83.26,2026-02-14,Address,43,3,0.997644,0,0,"CMS, FDA, HRSA, OS",NaN,NaN,180,469,154,low,[2026-03-11] Severity: low. VT score: 3. Top d...
1,101.168.57.163,2026-02-14,Address,2,3,0.999890,0,0,"CMS, FDA, OS, VA",Incident:INC9407577,NaN,180,590,458,medium,[2026-03-11] Severity: medium. VT score: 11. T...
2,101.71.130.99,2026-03-03,Address,139,3,0.992384,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",Incident:INC9270850,NaN,360,436,467,medium,[2026-03-11] Severity: medium. VT score: 11. T...
3,103.114.96.246,2026-02-26,Address,41,1,0.997753,1,0,"CMS, DHA, NIH, OS, VA",NaN,Mr Hamza Group,170,224,145,low,[2026-03-11] Severity: low. VT score: 2. Top d...
4,103.120.116.162,2026-03-11,Address,43,3,0.997644,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9385644,NaN,740,870,364,medium,[2026-03-11] Severity: medium. VT score: 6. To...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282,rosesci.com,2026-01-08,Host,17,0,0.999068,0,0,NIH,NaN,NaN,170,351,30,low,Severity: low. Could not pull VT score to calc...
2283,w-2@doculivery.com,2026-03-04,EmailAddress,2,4,0.999890,1,0,HHS,Incident:547380,NaN,170,224,29,low,[2026-03-06] Severity: low. VT score not avail...
2284,link.edgepilot.com,2026-03-05,Host,74,1,0.995945,0,0,"CMS, NIH",NaN,NaN,180,229,28,low,[2026-03-06] Severity: low. VT score not avail...
2285,52.87.206.242,2026-01-08,Address,1,1,0.999945,1,0,IHS,NaN,NaN,170,224,27,low,Severity: low. VT score: 0. Top drivers: TC ra...


In [4]:
merged = grouped.merge(tas_df, on="Indicator", how="left")

# Drop rows where all TAS columns are missing
tas_cols = [c for c in merged.columns if c not in grouped.columns]
merged_nonnull = merged.dropna(subset=tas_cols, how="all").copy()

# Fill missing incidents/events and threat actor fields with text

# Columns related to incidents/events
incident_cols = [c for c in merged_nonnull.columns 
                 if "incident" in c.lower() or "event" in c.lower()]

# Columns related to threat actors
threat_actor_cols = [c for c in merged_nonnull.columns 
                     if "threat actor" in c.lower() or "actor" in c.lower()]

merged_nonnull.loc[:, incident_cols] = merged_nonnull[incident_cols].fillna("No known incidents/events")
merged_nonnull.loc[:, threat_actor_cols] = merged_nonnull[threat_actor_cols].fillna("No known threat actors")

merged_nonnull

,Indicator,Frequency (7d),Frequency (30d),Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day,Frequency (1d),Probability: 1-Day,Last Observed,...,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0,1,11.20,22.60,32.99,36.37,0,0.42,2026-01-08,...,1.0,0.0,VA,Incident:30246,No known threat actors,810.0,905.0,177.0,low,Severity: low. Could not pull VT score to calc...
1,1-you.njalla.no,0,2,24.55,41.27,63.04,49.73,0,9.32,2026-02-03,...,0.0,0.0,"NIH, VA",No known incidents/events,Wicked Panda,180.0,416.0,312.0,medium,[2026-03-05] Severity: medium. VT score not av...
4,1.234.83.26,3,13,66.81,75.04,88.05,75.68,0,34.68,2026-02-14,...,0.0,0.0,"CMS, FDA, HRSA, OS",No known incidents/events,No known threat actors,180.0,469.0,154.0,low,[2026-03-11] Severity: low. VT score: 3. Top d...
5,1.4.195.14,0,2,23.35,35.03,54.08,50.42,0,5.77,2026-01-17,...,1.0,0.0,"CMS, VA",No known incidents/events,No known threat actors,180.0,229.0,134.0,low,Severity: low. VT score: 1. Top drivers: TOR a...
12,101.168.57.163,0,2,18.06,40.40,81.88,65.06,0,5.51,2026-02-14,...,0.0,0.0,"CMS, FDA, OS, VA",Incident:INC9407577,No known threat actors,180.0,590.0,458.0,medium,[2026-03-11] Severity: medium. VT score: 11. T...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5762,www.deepseek.com,2,11,63.03,72.67,81.67,74.15,0,23.93,2026-03-10,...,0.0,0.0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",No known incidents/events,No known threat actors,180.0,469.0,50.0,low,[2026-03-11] Severity: low. VT score not avail...
5763,www.deepseek.com.cdn.cloudflare.net,1,4,23.25,32.04,44.13,40.74,0,4.17,2025-11-28,...,0.0,0.0,"CMS, DHA, IHS, NIH, VA",No known incidents/events,No known threat actors,170.0,464.0,293.0,medium,Severity: medium. Could not pull VT score to c...
5792,www.prosinthecity.com,0,1,8.38,16.41,32.61,33.86,0,3.36,2025-11-12,...,0.0,0.0,"CMS, DHA, NIH, OS",Incident:6755399448004157;Incident:546481,No known threat actors,170.0,363.0,417.0,medium,Severity: medium. Could not pull VT score to c...
5795,www.sthda.com,1,8,45.24,58.82,69.60,63.92,0,11.00,2026-03-03,...,0.0,0.0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:331131,No known threat actors,180.0,368.0,153.0,low,[2026-03-11] Severity: low. VT score not avail...


In [20]:
from datetime import datetime, timedelta

# Ensure 'Last Observed' or 'Last Observed Indicator' column is present
date_cols = [c for c in merged_nonnull.columns if 'last observed' in c.lower()]
if not date_cols:
    raise ValueError("No column for last observed date found. Check for date column naming.")
last_observed_col = date_cols[0]

# Convert to datetime if not already
merged_nonnull[last_observed_col] = pd.to_datetime(merged_nonnull[last_observed_col], errors="coerce")

cutoff_date = datetime.now() - timedelta(hours=48)
recent_df = merged_nonnull[merged_nonnull[last_observed_col] >= cutoff_date]

if "HTOC Threat Score" in recent_df.columns:
    score_col = "HTOC Threat Score"
else:
    score_cols = [c for c in recent_df.columns if "threat score" in c.lower()]
    if not score_cols:
        raise ValueError("No Threat Score column found in filtered DataFrame.")
    score_col = score_cols[0]

top_10 = recent_df.sort_values(by=score_col, ascending=False).head(10)
top_10 = top_10[["Indicator", score_col, last_observed_col]]
top_10

,Indicator,HTOC Threat Score,Last Observed
3726,45.148.10.141,820.0,2026-03-11
1651,176.65.134.22,748.0,2026-03-11
3095,213.209.159.158,692.0,2026-03-11
101,103.143.10.79,676.0,2026-03-11
773,139.19.117.131,630.0,2026-03-11
2743,2.57.121.112,629.0,2026-03-11
1328,165.154.10.165,620.0,2026-03-11
2745,2.57.121.25,609.0,2026-03-11
4720,80.94.92.171,601.0,2026-03-11
3019,207.90.244.3,599.0,2026-03-11


In [27]:
# Table: top-10 recent indicators with their probabilities

prob_cols = [
    "Indicator",
    score_col,                 # e.g. "HTOC Threat Score"
    "Probability: 1-Day",
    "Probability: 7-Day",
    "Probability: 14-Day",
    "Probability: 30-Day",
    "Probability: 45-Day",
]

# Keep only columns that actually exist
prob_cols = [c for c in prob_cols if c in recent_df.columns]

top_10_probabilities = (
    recent_df[recent_df["Indicator"].isin(top_10["Indicator"])]
    [prob_cols]
    .drop_duplicates(subset=["Indicator"])
    .sort_values(by=score_col, ascending=False)
    .reset_index(drop=True)
)

top_10_probabilities

,Indicator,HTOC Threat Score,Probability: 1-Day,Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day
0,45.148.10.141,820.0,44.32,75.00,79.85,86.96,75.06
1,176.65.134.22,748.0,52.04,83.52,90.32,95.74,77.66
2,213.209.159.158,692.0,40.73,86.36,95.77,99.51,76.29
3,103.143.10.79,676.0,52.43,81.07,94.17,99.39,75.70
4,139.19.117.131,630.0,66.83,97.86,99.34,99.96,91.38
5,2.57.121.112,629.0,56.31,98.29,99.71,99.96,89.21
6,165.154.10.165,620.0,52.39,92.44,97.28,99.34,88.66
7,2.57.121.25,609.0,59.85,99.05,99.60,99.89,88.83
8,80.94.92.171,601.0,69.53,94.35,96.24,98.00,84.56
9,207.90.244.3,599.0,74.97,98.46,99.48,99.89,82.10


Threat Recurrence Risk Idex

In [5]:
# Total TRRI scores across all (assumed active) indicators
# Expected Value = Impact × Likelihood
trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

total_trri = {col: merged_nonnull[col].sum(skipna=True) for col in trri_cols}

total_trri

{}

In [6]:
import pandas as pd

preferred_score_col = "HTOC Threat Score"

if preferred_score_col in merged_nonnull.columns:
    score_col = preferred_score_col
else:
    score_cols = [c for c in merged_nonnull.columns if "threat score" in c.lower()]
    if not score_cols:
        raise ValueError("No Threat Score column found; check TAS column names.")
    score_col = score_cols[0]

merged_nonnull[score_col] = pd.to_numeric(merged_nonnull[score_col], errors="coerce")

prob_map = {
    "1d": "Probability: 1-Day",
    "7d": "Probability: 7-Day",
    "14d": "Probability: 14-Day",
    "30d": "Probability: 30-Day",
    "45d": "Probability: 45-Day",
}

for suffix, pcol in prob_map.items():
    if pcol not in merged_nonnull.columns:
        continue

    merged_nonnull[pcol] = pd.to_numeric(merged_nonnull[pcol], errors="coerce")

    max_prob = merged_nonnull[pcol].max(skipna=True)

    # If max > 1, assume percentages like 12, 25, 68
    if pd.notna(max_prob) and max_prob > 1:
        prob_series = merged_nonnull[pcol] / 100.0
    else:
        prob_series = merged_nonnull[pcol]

    merged_nonnull[f"TRRI_{suffix}"] = (
        merged_nonnull[score_col] * prob_series
    ).round(1)

display_cols = ["Indicator", score_col] + [f"TRRI_{k}" for k in prob_map if f"TRRI_{k}" in merged_nonnull.columns]
merged_nonnull[display_cols]

,Indicator,HTOC Threat Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,177.0,0.7,19.8,40.0,58.4,64.4
1,1-you.njalla.no,312.0,29.1,76.6,128.8,196.7,155.2
4,1.234.83.26,154.0,53.4,102.9,115.6,135.6,116.5
5,1.4.195.14,134.0,7.7,31.3,46.9,72.5,67.6
12,101.168.57.163,458.0,25.2,82.7,185.0,375.0,298.0
...,...,...,...,...,...,...,...
5762,www.deepseek.com,50.0,12.0,31.5,36.3,40.8,37.1
5763,www.deepseek.com.cdn.cloudflare.net,293.0,12.2,68.1,93.9,129.3,119.4
5792,www.prosinthecity.com,417.0,14.0,34.9,68.4,136.0,141.2
5795,www.sthda.com,153.0,16.8,69.2,90.0,106.5,97.8


In [7]:
# Step 5 — Create TRRI risk zones per horizon using percentiles

import pandas as pd

# Percentile cut-points:
# 0–40%   -> Low
# 40–70%  -> Moderate
# 70–90%  -> High
# 90–100% -> Critical
quantiles = [0.0, 0.4, 0.7, 0.9, 1.0]
labels = [
    "Low Recurrence Risk",        # bottom 40%
    "Moderate Recurrence Risk",   # next 30%
    "High Recurrence Risk",       # next 20%
    "Critical Recurrence Risk",   # top 10%
]

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

for col in trri_cols:
    risk_col = col.replace("TRRI_", "Risk_")
    # Use qcut on the TRRI values to assign percentile-based risk
    merged_nonnull[risk_col] = pd.qcut(
        merged_nonnull[col],
        q=quantiles,
        labels=labels,
        duplicates="drop"  # handle cases where some bins collapse
    )

# Quick look at indicator, Threat Score, TRRI, and risk zones
cols_to_show = (
    ["Indicator", "HTOC Threat Score"]
    + trri_cols
    + [c for c in merged_nonnull.columns if c.startswith("Risk_")]
)
merged_nonnull[cols_to_show]

,Indicator,HTOC Threat Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d,Risk_1d,Risk_7d,Risk_14d,Risk_30d,Risk_45d
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,177.0,0.7,19.8,40.0,58.4,64.4,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
1,1-you.njalla.no,312.0,29.1,76.6,128.8,196.7,155.2,Moderate Recurrence Risk,Low Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk
4,1.234.83.26,154.0,53.4,102.9,115.6,135.6,116.5,Moderate Recurrence Risk,Moderate Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5,1.4.195.14,134.0,7.7,31.3,46.9,72.5,67.6,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
12,101.168.57.163,458.0,25.2,82.7,185.0,375.0,298.0,Moderate Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk,High Recurrence Risk,High Recurrence Risk
...,...,...,...,...,...,...,...,...,...,...,...,...
5762,www.deepseek.com,50.0,12.0,31.5,36.3,40.8,37.1,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5763,www.deepseek.com.cdn.cloudflare.net,293.0,12.2,68.1,93.9,129.3,119.4,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5792,www.prosinthecity.com,417.0,14.0,34.9,68.4,136.0,141.2,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5795,www.sthda.com,153.0,16.8,69.2,90.0,106.5,97.8,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk


In [25]:
# Indicators from your 48-hour / top-threat subset
top_inds = top_10["Indicator"].unique()

# Filter the full merged DF to just those indicators
recent_top_trri = merged_nonnull[merged_nonnull["Indicator"].isin(top_inds)].copy()

# Build a TRRI-focused view, sorted by HTOC Threat Score
cols = [
    "Indicator",
    "HTOC Threat Score",
    "TRRI_1d",
    "TRRI_7d",
    "TRRI_14d",
    "TRRI_30d",
    "TRRI_45d",
    "Risk_1d",
    "Risk_7d",
    "Risk_14d",
    "Risk_30d",
    "Risk_45d",
]
existing_cols = [c for c in cols if c in recent_top_trri.columns]

recent_top_trri = (
    recent_top_trri[existing_cols]
    .sort_values("HTOC Threat Score", ascending=False)
    .reset_index(drop=True)
)

recent_top_trri

,Indicator,HTOC Threat Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d,Risk_1d,Risk_7d,Risk_14d,Risk_30d,Risk_45d
0,45.148.10.141,820.0,363.4,615.0,654.8,713.1,615.5,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
1,176.65.134.22,748.0,389.3,624.7,675.6,716.1,580.9,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
2,213.209.159.158,692.0,281.9,597.6,662.7,688.6,527.9,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
3,103.143.10.79,676.0,354.4,548.0,636.6,671.9,511.7,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
4,139.19.117.131,630.0,421.0,616.5,625.8,629.7,575.7,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
5,2.57.121.112,629.0,354.2,618.2,627.2,628.7,561.1,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
6,165.154.10.165,620.0,324.8,573.1,603.1,615.9,549.7,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
7,2.57.121.25,609.0,364.5,603.2,606.6,608.3,541.0,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
8,80.94.92.171,601.0,417.9,567.0,578.4,589.0,508.2,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
9,207.90.244.3,599.0,449.1,589.8,595.9,598.3,491.8,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk


In [26]:
# Table: top-10 recent indicators with their explanations
explanation_cols = ["Indicator", score_col, "Explanation"]
explanation_cols = [c for c in explanation_cols if c in recent_df.columns]

top_10_explanations = (
    recent_df[recent_df["Indicator"].isin(top_10["Indicator"])]
    [explanation_cols]
    .drop_duplicates(subset=["Indicator"])
    .sort_values(by=score_col, ascending=False)
    .reset_index(drop=True)
)

top_10_explanations

,Indicator,HTOC Threat Score,Explanation
0,45.148.10.141,820.0,[2026-03-11] Severity: critical. VT score: 15....
1,176.65.134.22,748.0,[2026-03-11] Severity: high. VT score: 21. Top...
2,213.209.159.158,692.0,[2026-03-11] Severity: high. VT score: 18. Top...
3,103.143.10.79,676.0,[2026-03-11] Severity: high. VT score: 17. Top...
4,139.19.117.131,630.0,[2026-03-11] Severity: high. VT score: 15. Top...
5,2.57.121.112,629.0,[2026-03-11] Severity: high. VT score: 16. Top...
6,165.154.10.165,620.0,[2026-03-11] Severity: high. VT score: 15. Top...
7,2.57.121.25,609.0,[2026-03-11] Severity: high. VT score: 15. Top...
8,80.94.92.171,601.0,[2026-03-11] Severity: high. VT score: 17. Top...
9,207.90.244.3,599.0,[2026-03-11] Severity: high. VT score: 14. Top...


import os

trri_dir = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI"
os.makedirs(trri_dir, exist_ok=True)

out_path = os.path.join(trri_dir, "TRRI_merged_nonnull_2025-06-16.xlsx")  # adjust date/name as needed
merged_nonnull.to_excel(out_path, index=False)

out_path

In [8]:

# DataFrames of the top 10 indicators for each TRRI column, along with their Threat Score
top_10_trri_dfs = {}

for col in trri_cols:
    # Get top 10 by TRRI value, include Indicator and Threat Score
    df_top10 = (
        merged_nonnull[["Indicator", "HTOC Threat Score", col]]
        .sort_values(col, ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    top_10_trri_dfs[col] = df_top10
    print(f"\n=== Top 10 for {col} ===")
    display(df_top10)





=== Top 10 for TRRI_1d ===


,Indicator,HTOC Threat Score,TRRI_1d
0,207.90.244.3,599.0,449.1
1,198.235.24.186,577.0,436.3
2,205.210.31.163,584.0,425.4
3,139.19.117.131,630.0,421.0
4,80.94.92.171,601.0,417.9
5,64.225.74.178,579.0,415.4
6,123.58.207.81,596.0,410.1
7,147.185.132.58,541.0,407.5
8,176.65.134.22,748.0,389.3
9,205.210.31.47,499.0,383.8



=== Top 10 for TRRI_7d ===


,Indicator,HTOC Threat Score,TRRI_7d
0,176.65.134.22,748.0,624.7
1,2.57.121.112,629.0,618.2
2,139.19.117.131,630.0,616.5
3,45.148.10.141,820.0,615.0
4,2.57.121.25,609.0,603.2
5,213.209.159.158,692.0,597.6
6,207.90.244.3,599.0,589.8
7,123.58.207.81,596.0,583.5
8,205.210.31.163,584.0,576.2
9,165.154.10.165,620.0,573.1



=== Top 10 for TRRI_14d ===


,Indicator,HTOC Threat Score,TRRI_14d
0,176.65.134.22,748.0,675.6
1,213.209.159.158,692.0,662.7
2,45.148.10.141,820.0,654.8
3,103.143.10.79,676.0,636.6
4,2.57.121.112,629.0,627.2
5,139.19.117.131,630.0,625.8
6,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,980.0,610.9
7,2.57.121.25,609.0,606.6
8,45.9.148.129,633.0,603.7
9,165.154.10.165,620.0,603.1



=== Top 10 for TRRI_30d ===


,Indicator,HTOC Threat Score,TRRI_30d
0,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,980.0,736.2
1,176.65.134.22,748.0,716.1
2,45.148.10.141,820.0,713.1
3,213.209.159.158,692.0,688.6
4,103.143.10.79,676.0,671.9
5,139.19.117.131,630.0,629.7
6,45.9.148.129,633.0,629.1
7,2.57.121.112,629.0,628.7
8,165.154.10.165,620.0,615.9
9,2.57.121.25,609.0,608.3



=== Top 10 for TRRI_45d ===


,Indicator,HTOC Threat Score,TRRI_45d
0,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,980.0,672.1
1,45.148.10.141,820.0,615.5
2,176.65.134.22,748.0,580.9
3,139.19.117.131,630.0,575.7
4,2.57.121.112,629.0,561.1
5,123.58.207.81,596.0,552.6
6,165.154.10.165,620.0,549.7
7,2.57.121.25,609.0,541.0
8,61.153.184.151,646.0,540.5
9,213.209.159.158,692.0,527.9


In [9]:
# Total TRRI scores across all (assumed active) indicators

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

total_trri = {col: merged_nonnull[col].sum(skipna=True) for col in trri_cols}

# Turn into a clean DataFrame
total_trri_df = (
    pd.DataFrame(list(total_trri.items()), columns=["TRRI_Horizon", "Total_TRRI"])
    .sort_values("TRRI_Horizon")
    .reset_index(drop=True)
)

total_trri_df

,TRRI_Horizon,Total_TRRI
0,TRRI_14d,408366.0
1,TRRI_1d,199711.7
2,TRRI_30d,475024.6
3,TRRI_45d,419845.0
4,TRRI_7d,349392.9


# Save current TRRI totals to CSV
out_path = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI\TRRI_totals_2025-06-16.csv"  # adjust name/date as needed
total_trri_df.to_csv(out_path, index=False)
out_path

import os
import glob
import pandas as pd

trri_dir = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI"

# Get all TRRI total CSVs in the folder
csv_files = glob.glob(os.path.join(trri_dir, "TRRI_totals_*.csv"))
if len(csv_files) < 2:
    raise ValueError("Need at least two TRRI_totals_*.csv files to compare.")

# Sort by filename (assumes date is in the name and sortable, e.g. TRRI_totals_2025-06-16.csv)
csv_files_sorted = sorted(csv_files)

prev_path = csv_files_sorted[-2]  # previous file
curr_path = csv_files_sorted[-1]  # current file

prev_df = pd.read_csv(prev_path)
curr_df = pd.read_csv(curr_path)

compare = prev_df.merge(
    curr_df,
    on="TRRI_Horizon",
    suffixes=("_prev", "_curr")
)

compare["Pct_Change"] = (
    (compare["Total_TRRI_curr"] - compare["Total_TRRI_prev"])
    / compare["Total_TRRI_prev"]
) * 100

compare